In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
# ṭurn off AQE
spark.conf.set("spark.sql.adaptive.enabled", "false")

spark.conf.set("spark.sql.codegen.wholeStage", "false")

In [0]:
# Example 1
df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")

df.write.mode("overwrite").format("noop").save()

df.explain()

+-----------+-----------------+------------+-----+------------+--------+
|customer_id|            email|        city|state|   full_name|  domain|
+-----------+-----------------+------------+-----+------------+--------+
|     C00001|rushjeff@ryan.org|Johnsonmouth|   MS|Emily Mooney|ryan.org|
+-----------+-----------------+------------+-----+------------+--------+

== Physical Plan ==
Project [customer_id#324, email#325, city#326, state#327, full_name#328, domain#329]
+- Filter if (isnotnull(_databricks_internal_edge_computed_column_skip_row#338)) (_databricks_internal_edge_computed_column_skip_row#338 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType))
   +- FileScan parquet azure_retail_project_catalog.silver.silver_customers[customer_id#324,email#325,city#326,state#327,full_name#328,domain#329,_databricks_internal_edge_computed_column_skip_row#338] Batched: false, DataFilters: [], Format: Parquet, Location: PreparedDeltaFileIndex

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_1.png)

In [0]:
# Example 2
df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")

df = df.filter(col("state") == "LA")

df = df.select("customer_id", "full_name", "state")

df.write.mode("overwrite").format("noop").save()

df.explain()

+-----------+-----------+-----+
|customer_id|  full_name|state|
+-----------+-----------+-----+
|     C00003|Craig Hayes|   LA|
+-----------+-----------+-----+

== Physical Plan ==
Project [customer_id#399, full_name#403, state#402]
+- Filter ((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#420)) (_databricks_internal_edge_computed_column_skip_row#420 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#402)) AND (state#402 = LA))
   +- FileScan parquet azure_retail_project_catalog.silver.silver_customers[customer_id#399,state#402,full_name#403,_databricks_internal_edge_computed_column_skip_row#420] Batched: false, DataFilters: [isnotnull(state#402), (state#402 = LA)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://azure-retail-project@sapracticeav.dfs.core.windows.net/si..., PartitionFilters: [], PushedFilters: [IsNotNull(state), EqualTo(state,LA)], ReadSchema: struct<custo

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_2.png)

In [0]:
# Example 3:
df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")

df2 = df.filter(df.state.isNotNull())

df3 = df2.groupBy("state").count()

df3.limit(1).display()

df3.explain()

state,count
SC,42


== Physical Plan ==
HashAggregate(keys=[state#521], functions=[finalmerge_count(merge count#527L) AS count(1)#524L])
+- Exchange hashpartitioning(state#521, 200), ENSURE_REQUIREMENTS, [plan_id=169]
   +- HashAggregate(keys=[state#521], functions=[partial_count(1) AS count#527L])
      +- Project [state#521]
         +- Filter (if (isnotnull(_databricks_internal_edge_computed_column_skip_row#538)) (_databricks_internal_edge_computed_column_skip_row#538 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#521))
            +- FileScan parquet azure_retail_project_catalog.silver.silver_customers[state#521,_databricks_internal_edge_computed_column_skip_row#538] Batched: false, DataFilters: [isnotnull(state#521)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://azure-retail-project@sapracticeav.dfs.core.windows.net/si..., PartitionFilters: [], PushedFilters: [IsNotNull(state)], ReadSchema: struct<s

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_3.png)

In [0]:
# Example 4:
customers_la = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
customers_la = customers_la.filter("state = 'LA'")
customers_la = customers_la.repartition(6)

customers_la.write.mode("overwrite").format("noop").save()

customers_la.explain()

== Physical Plan ==
Exchange RoundRobinPartitioning(6), REPARTITION_BY_NUM, [plan_id=271]
+- Project [customer_id#728, email#729, city#730, state#731, full_name#732, domain#733]
   +- Filter ((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#749)) (_databricks_internal_edge_computed_column_skip_row#749 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#731)) AND (state#731 = LA))
      +- FileScan parquet azure_retail_project_catalog.silver.silver_customers[customer_id#728,email#729,city#730,state#731,full_name#732,domain#733,_databricks_internal_edge_computed_column_skip_row#749] Batched: false, DataFilters: [isnotnull(state#731), (state#731 = LA)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://azure-retail-project@sapracticeav.dfs.core.windows.net/si..., PartitionFilters: [], PushedFilters: [IsNotNull(state), EqualTo(state,LA)], ReadSchema: struct<customer_id:string,email

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_4.png)

In [0]:
# Example 5:
df = spark.read.table(
    "azure_retail_project_catalog.silver.silver_customers"
)

df2 = df.filter(df.state.isNotNull())

df3 = df2.groupBy("state").count()

df4 = df3.repartition(4)

df4.write.mode("overwrite").format("noop").save()

df4.explain()

== Physical Plan ==
Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=346]
+- HashAggregate(keys=[state#861], functions=[finalmerge_count(merge count#867L) AS count(1)#864L])
   +- Exchange hashpartitioning(state#861, 200), ENSURE_REQUIREMENTS, [plan_id=344]
      +- HashAggregate(keys=[state#861], functions=[partial_count(1) AS count#867L])
         +- Project [state#861]
            +- Filter (if (isnotnull(_databricks_internal_edge_computed_column_skip_row#878)) (_databricks_internal_edge_computed_column_skip_row#878 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#861))
               +- FileScan parquet azure_retail_project_catalog.silver.silver_customers[state#861,_databricks_internal_edge_computed_column_skip_row#878] Batched: false, DataFilters: [isnotnull(state#861)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://azure-retail-project@sapracticeav.dfs.core.windows.

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_5.png)

In [0]:
# Example 6:
df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")

ca_cust_df = df.filter(col("state") == "CA")
la_cust_df = df.filter(col("state") == "LA")

ca_cust_df = ca_cust_df.select("customer_id", "email", "city", "state", "full_name", "domain")
ca_grouped_df = ca_cust_df.groupBy("city").count()


la_cust_df = la_cust_df.select("full_name", "state")
la_grouped_df = ca_cust_df.groupBy("state").count()

# action 1: Job 1
ca_grouped_df.write.mode("overwrite").format("noop").save()
ca_grouped_df.explain()

# action 2: Job 2
la_grouped_df.write.mode("overwrite").format("noop").save()


== Physical Plan ==
HashAggregate(keys=[city#1001], functions=[finalmerge_count(merge count#1008L) AS count(1)#1005L])
+- Exchange hashpartitioning(city#1001, 200), ENSURE_REQUIREMENTS, [plan_id=413]
   +- HashAggregate(keys=[city#1001], functions=[partial_count(1) AS count#1008L])
      +- Project [city#1001]
         +- Filter ((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#1023)) (_databricks_internal_edge_computed_column_skip_row#1023 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#1002)) AND (state#1002 = CA))
            +- FileScan parquet azure_retail_project_catalog.silver.silver_customers[city#1001,state#1002,_databricks_internal_edge_computed_column_skip_row#1023] Batched: false, DataFilters: [isnotnull(state#1002), (state#1002 = CA)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://azure-retail-project@sapracticeav.dfs.core.windows.net/si..., PartitionFilter

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_6.png)

In [0]:
# Example 7:
cust_df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
cust_df = (
cust_df
    .filter(col("state") == "LA")
    .withColumn("flag", lit(1))
)

grouped_cust_df = cust_df.groupBy("state").count()
# Action 1 = Job 1
grouped_cust_df.write.mode("overwrite").format("noop").save()
grouped_cust_df.explain()


orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter(col("quantity") < 5)

# Action 2 = Job 2
grouped_orders_df = orders_df.groupBy("order_id").count()

# Action 3 = Job 3
grouped_orders_df.write.mode("overwrite").format("noop").save()


== Physical Plan ==
HashAggregate(keys=[state#1270], functions=[finalmerge_count(merge count#1279L) AS count(1)#1277L])
+- Exchange hashpartitioning(state#1270, 200), ENSURE_REQUIREMENTS, [plan_id=512]
   +- HashAggregate(keys=[state#1270], functions=[partial_count(1) AS count#1279L])
      +- Project [state#1270]
         +- Filter ((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#1294)) (_databricks_internal_edge_computed_column_skip_row#1294 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#1270)) AND (state#1270 = LA))
            +- FileScan parquet azure_retail_project_catalog.silver.silver_customers[state#1270,_databricks_internal_edge_computed_column_skip_row#1294] Batched: false, DataFilters: [isnotnull(state#1270), (state#1270 = LA)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://azure-retail-project@sapracticeav.dfs.core.windows.net/si..., PartitionFilters: [],

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_7.png)

In [0]:
# Example 8:
cust_df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
cust_df = cust_df.filter("state = 'LA'")

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity < 5")

# using shuffle sort merge join
joined_df = cust_df.hint("merge").join(orders_df, "customer_id")

joined_df.write.mode("overwrite").format("noop").save()

joined_df.explain()

# Note: Behavior of Spark will be different with different joins

== Physical Plan ==
Project [customer_id#1596, email#1597, city#1598, state#1599, full_name#1600, domain#1601, order_id#1608, product_id#1610, order_date#1611, quantity#1612, total_amount#1613]
+- SortMergeJoin [customer_id#1596], [customer_id#1609], Inner
   :- Sort [customer_id#1596 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#1596, 200), ENSURE_REQUIREMENTS, [plan_id=660]
   :     +- Project [customer_id#1596, email#1597, city#1598, state#1599, full_name#1600, domain#1601]
   :        +- Filter (((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#1646)) (_databricks_internal_edge_computed_column_skip_row#1646 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#1599)) AND (state#1599 = LA)) AND isnotnull(customer_id#1596))
   :           +- FileScan parquet azure_retail_project_catalog.silver.silver_customers[customer_id#1596,email#1597,city#1598,state#1599,full_nam

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_8.png)

In [0]:
# Example 9:
cust_df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
cust_df = cust_df.filter("state = 'LA'")

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity < 5")

# using broadcast hash join
joined_df = cust_df.hint("broadcast").join(orders_df, "customer_id")

joined_df.write.mode("overwrite").format("noop").save()

joined_df.explain()

# Note: Behavior of Spark will be different with different joins

== Physical Plan ==
Project [customer_id#2380, email#2381, city#2382, state#2383, full_name#2384, domain#2385, order_id#2386, product_id#2388, order_date#2389, quantity#2390, total_amount#2391]
+- BroadcastHashJoin [customer_id#2380], [customer_id#2387], Inner, BuildLeft, false, false
   :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=962]
   :  +- Project [customer_id#2380, email#2381, city#2382, state#2383, full_name#2384, domain#2385]
   :     +- Filter (((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#2424)) (_databricks_internal_edge_computed_column_skip_row#2424 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#2383)) AND (state#2383 = LA)) AND isnotnull(customer_id#2380))
   :        +- FileScan parquet azure_retail_project_catalog.silver.silver_customers[customer_id#2380,email#2381,city#2382,state#2383,full_name#2384,domain#2385,_databri

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_9.png)

In [0]:
# Example 10:
cust_df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
cust_df = cust_df.filter("state = 'LA'")


orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity < 5")

joined_df = cust_df.hint("merge").join(orders_df, "customer_id")

result_df = joined_df.groupBy("state").count()

result_df.write.mode("overwrite").format("noop").save()

result_df.explain()

== Physical Plan ==
HashAggregate(keys=[state#2614], functions=[finalmerge_count(merge count#2633L) AS count(1)#2629L])
+- Exchange hashpartitioning(state#2614, 200), ENSURE_REQUIREMENTS, [plan_id=1098]
   +- HashAggregate(keys=[state#2614], functions=[partial_count(1) AS count#2633L])
      +- Project [state#2614]
         +- SortMergeJoin [customer_id#2611], [customer_id#2624], Inner
            :- Sort [customer_id#2611 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(customer_id#2611, 200), ENSURE_REQUIREMENTS, [plan_id=1090]
            :     +- Project [customer_id#2611, state#2614]
            :        +- Filter (((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#2665)) (_databricks_internal_edge_computed_column_skip_row#2665 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#2614)) AND (state#2614 = LA)) AND isnotnull(customer_id#2611))
            :           +- F

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_10.png)

In [0]:
# Example 11:
cust_df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
cust_df = cust_df.filter("state = 'LA'")
cust_df = cust_df.repartition(4, "customer_id")

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity < 5")
orders_df = orders_df.repartition(4, "customer_id")

# Since we already repartitioned the data on customer_id, so spark will not shuffle the data while join

joined_df = cust_df.hint("merge").join(orders_df, "customer_id")

result_df = joined_df.groupBy("state").count()

result_df.write.mode("overwrite").format("noop").save()

result_df.explain()


== Physical Plan ==
HashAggregate(keys=[state#2885], functions=[finalmerge_count(merge count#2904L) AS count(1)#2900L])
+- Exchange hashpartitioning(state#2885, 200), ENSURE_REQUIREMENTS, [plan_id=1272]
   +- HashAggregate(keys=[state#2885], functions=[partial_count(1) AS count#2904L])
      +- Project [state#2885]
         +- SortMergeJoin [customer_id#2882], [customer_id#2895], Inner
            :- Sort [customer_id#2882 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(customer_id#2882, 200), REPARTITION_BY_NUM, [plan_id=1264]
            :     +- Project [customer_id#2882, state#2885]
            :        +- Filter (((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#2936)) (_databricks_internal_edge_computed_column_skip_row#2936 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#2885)) AND (state#2885 = LA)) AND isnotnull(customer_id#2882))
            :           +- Fi

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_11.png)

In [0]:
# Example 12:
cust_df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
cust_df = cust_df.filter("state = 'LA'")
cust_df = cust_df.repartition(4, "city")

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity < 5")
orders_df = orders_df.repartition(4, "product_id")

# in this case we repartitoned the data on different columns, so spark will shuffle the data while join 

joined_df = cust_df.hint("merge").join(orders_df, "customer_id")

result_df = joined_df.groupBy("state").count()

result_df.write.mode("overwrite").format("noop").save()

result_df.explain()


== Physical Plan ==
HashAggregate(keys=[state#3160], functions=[finalmerge_count(merge count#3173L) AS count(1)#3169L])
+- Exchange hashpartitioning(state#3160, 200), ENSURE_REQUIREMENTS, [plan_id=1472]
   +- HashAggregate(keys=[state#3160], functions=[partial_count(1) AS count#3173L])
      +- Project [state#3160]
         +- SortMergeJoin [customer_id#3157], [customer_id#3164], Inner
            :- Sort [customer_id#3157 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(customer_id#3157, 200), ENSURE_REQUIREMENTS, [plan_id=1464]
            :     +- Project [customer_id#3157, state#3160]
            :        +- Exchange hashpartitioning(city#3159, 4), REPARTITION_BY_NUM, [plan_id=1452]
            :           +- Project [customer_id#3157, city#3159, state#3160]
            :              +- Filter (((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#3205)) (_databricks_internal_edge_computed_column_skip_row#3205 = false) else isnotnull(raise_error(

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_12.png)

In [0]:
# Example 13:
cust_df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
cust_df = cust_df.filter("state = 'LA'")
cust_df = cust_df.repartition(4, "state")

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity < 5")
orders_df = orders_df.repartition(4, "product_id")

# in this case we already repartitioned the cust data on state so 
# spark will not shuffle the data again while groupBy

joined_df = cust_df.hint("merge").join(orders_df, "customer_id")

result_df = joined_df.groupBy("state").count()

result_df.write.mode("overwrite").format("noop").save()

result_df.explain()


== Physical Plan ==
HashAggregate(keys=[state#3425], functions=[finalmerge_count(merge count#3444L) AS count(1)#3440L])
+- Exchange hashpartitioning(state#3425, 200), ENSURE_REQUIREMENTS, [plan_id=1669]
   +- HashAggregate(keys=[state#3425], functions=[partial_count(1) AS count#3444L])
      +- Project [state#3425]
         +- SortMergeJoin [customer_id#3422], [customer_id#3435], Inner
            :- Sort [customer_id#3422 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(customer_id#3422, 200), ENSURE_REQUIREMENTS, [plan_id=1661]
            :     +- Project [customer_id#3422, state#3425]
            :        +- Filter (((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#3476)) (_databricks_internal_edge_computed_column_skip_row#3476 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(state#3425)) AND (state#3425 = LA)) AND isnotnull(customer_id#3422))
            :           +- F

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_13.png)

In [0]:
# Example 14:
cust_df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
cust_df = cust_df.filter("state IS NOT NULL")

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity > 1")
orders_df = orders_df.repartition(4,"customer_id")   # X

products_df = spark.read.table("azure_retail_project_catalog.silver.silver_products")
products_df = products_df.filter("price > 100")

joined_df = cust_df.hint("merge").join(orders_df,"customer_id")

joined_df = joined_df.hint("merge").join(products_df,"product_id")

result_df = joined_df.groupBy("state").count()

result_df.write.mode("overwrite").format("noop").save()

result_df.explain()

== Physical Plan ==
HashAggregate(keys=[state#3828], functions=[finalmerge_count(merge count#3856L) AS count(1)#3851L])
+- Exchange hashpartitioning(state#3828, 200), ENSURE_REQUIREMENTS, [plan_id=1927]
   +- HashAggregate(keys=[state#3828], functions=[partial_count(1) AS count#3856L])
      +- Project [state#3828]
         +- SortMergeJoin [product_id#3833], [product_id#3844], Inner
            :- Sort [product_id#3833 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#3833, 200), ENSURE_REQUIREMENTS, [plan_id=1919]
            :     +- Project [state#3828, product_id#3833]
            :        +- SortMergeJoin [customer_id#3825], [customer_id#3832], Inner
            :           :- Sort [customer_id#3825 ASC NULLS FIRST], false, 0
            :           :  +- Exchange hashpartitioning(customer_id#3825, 200), ENSURE_REQUIREMENTS, [plan_id=1911]
            :           :     +- Project [customer_id#3825, state#3828]
            :           :        +- Fi

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_14.png)

In [0]:
# Example 15:

cust_df = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
cust_df = cust_df.filter("state = 'LA'")
cust_df = cust_df.repartition(4,"customer_id")

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity > 2")

products_df = spark.read.table("azure_retail_project_catalog.silver.silver_products")
products_df = products_df.filter("price > 50")
products_df = products_df.repartition(4,"product_id")

# only one time shuffle will happen for customer_id and product_id

joined_df = cust_df.hint("merge").join(orders_df,"customer_id")
joined_df = joined_df.hint("merge").join(products_df,"product_id")

result_df = joined_df.groupBy("state").count()

result_df.write.mode("overwrite").format("noop").save()
result_df.explain()

== Physical Plan ==
HashAggregate(keys=[state#4211], functions=[finalmerge_count(merge count#4245L) AS count(1)#4240L])
+- Exchange hashpartitioning(state#4211, 200), ENSURE_REQUIREMENTS, [plan_id=2193]
   +- HashAggregate(keys=[state#4211], functions=[partial_count(1) AS count#4245L])
      +- Project [state#4211]
         +- SortMergeJoin [product_id#4229], [product_id#4233], Inner
            :- Sort [product_id#4229 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#4229, 200), ENSURE_REQUIREMENTS, [plan_id=2185]
            :     +- Project [state#4211, product_id#4229]
            :        +- SortMergeJoin [customer_id#4208], [customer_id#4228], Inner
            :           :- Sort [customer_id#4208 ASC NULLS FIRST], false, 0
            :           :  +- Exchange hashpartitioning(customer_id#4208, 200), REPARTITION_BY_NUM, [plan_id=2178]
            :           :     +- Project [customer_id#4208, state#4211]
            :           :        +- Fil

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_15.png)

In [0]:
# Example 16:
df1 = spark.read.table("azure_retail_project_catalog.practice.df1")
df2 = spark.read.table("azure_retail_project_catalog.practice.df2")

df3 = df1.hint("merge").join(df2, "id")

df3.write.mode("overwrite").format("noop").save()

df3.explain()

== Physical Plan ==
Project [id#4657, value#4658L, value#4660L]
+- SortMergeJoin [id#4657], [id#4659], Inner
   :- Sort [id#4657 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(id#4657, 200), ENSURE_REQUIREMENTS, [plan_id=2358]
   :     +- Project [id#4657, value#4658L]
   :        +- Filter (if (isnotnull(_databricks_internal_edge_computed_column_skip_row#4683)) (_databricks_internal_edge_computed_column_skip_row#4683 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(id#4657))
   :           +- FileScan parquet azure_retail_project_catalog.practice.df1[id#4657,value#4658L,_databricks_internal_edge_computed_column_skip_row#4683] Batched: false, DataFilters: [isnotnull(id#4657)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://azure-retail-project@sapracticeav.dfs.core.windows.net/ca..., PartitionFilters: [], PushedFilters: [IsNotNull(id)], ReadSchema: struct<id:string,value:bigint,_

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_16.png)

In [0]:
# Example 17:
df1 = spark.read.table("azure_retail_project_catalog.practice.df1")
df2 = spark.read.table("azure_retail_project_catalog.practice.df2")

df3 = df1.union(df2)

df3.write.mode("overwrite").format("noop").save()

df3.explain()

== Physical Plan ==
Union
:- Project [id#4754, value#4755L]
:  +- Filter if (isnotnull(_databricks_internal_edge_computed_column_skip_row#4774)) (_databricks_internal_edge_computed_column_skip_row#4774 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType))
:     +- FileScan parquet azure_retail_project_catalog.practice.df1[id#4754,value#4755L,_databricks_internal_edge_computed_column_skip_row#4774] Batched: false, DataFilters: [], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://azure-retail-project@sapracticeav.dfs.core.windows.net/ca..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<id:string,value:bigint,_databricks_internal_edge_computed_column_skip_row:boolean>
+- Project [id#4756, value#4757L]
   +- Filter if (isnotnull(_databricks_internal_edge_computed_column_skip_row#4775)) (_databricks_internal_edge_computed_column_skip_row#4775 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FI

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_17.png)

In [0]:
# Example 18:
customers_la = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
customers_la = customers_la.filter("state = 'LA'")
customers_la = customers_la.repartition(6)   

customers_tx = spark.read.table("azure_retail_project_catalog.silver.silver_customers")
customers_tx = customers_tx.filter("state = 'TX'")
customers_tx = customers_tx.repartition(6)

customers = customers_la.union(customers_tx)

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity > 2")

joined_df = customers.hint("merge").join(orders_df,"customer_id")

result_df = joined_df.groupBy("state").count()

result_df.write.mode("overwrite").format("noop").save()

result_df.explain()

== Physical Plan ==
HashAggregate(keys=[state#4960], functions=[finalmerge_count(merge count#4980L) AS count(1)#4975L])
+- Exchange hashpartitioning(state#4960, 200), ENSURE_REQUIREMENTS, [plan_id=2617]
   +- HashAggregate(keys=[state#4960], functions=[partial_count(1) AS count#4980L])
      +- Project [state#4960]
         +- SortMergeJoin [customer_id#4957], [customer_id#4970], Inner
            :- Sort [customer_id#4957 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(customer_id#4957, 200), ENSURE_REQUIREMENTS, [plan_id=2609]
            :     +- Union
            :        :- Exchange RoundRobinPartitioning(6), REPARTITION_BY_NUM, [plan_id=2595]
            :        :  +- Project [customer_id#4957, state#4960]
            :        :     +- Filter (((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#5028)) (_databricks_internal_edge_computed_column_skip_row#5028 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], v

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_18.png)

In [0]:
# Example 19:
# Window Functions - Complete Example

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity > 1")

window_spec = Window.partitionBy("customer_id").orderBy("order_date")

result_df = orders_df.withColumn("row_num",row_number().over(window_spec))
result_df = result_df.withColumn("running_qty",sum("quantity").over(window_spec))
result_df = result_df.withColumn("previous_qty",lag("quantity").over(window_spec))

result_df.write.mode("overwrite").format("noop").save()

result_df.explain()

== Physical Plan ==
Window [order_id#6661, customer_id#6662, product_id#6663, order_date#6664, quantity#6665, total_amount#6666, row_number() windowspecdefinition(customer_id#6662, order_date#6664 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS row_num#6669, sum(quantity#6665) windowspecdefinition(customer_id#6662, order_date#6664 ASC NULLS FIRST, specifiedwindowframe(RangeFrame, unboundedpreceding$(), currentrow$())) AS running_qty#6672L, lag(quantity#6665, -1, null) windowspecdefinition(customer_id#6662, order_date#6664 ASC NULLS FIRST, specifiedwindowframe(RowFrame, -1, -1)) AS previous_qty#6675], [customer_id#6662], [order_date#6664 ASC NULLS FIRST]
+- Sort [customer_id#6662 ASC NULLS FIRST, order_date#6664 ASC NULLS FIRST], false, 0
   +- Exchange hashpartitioning(customer_id#6662, 200), ENSURE_REQUIREMENTS, [plan_id=2957]
      +- Project [order_id#6661, customer_id#6662, product_id#6663, order_date#6664, quantity#6665, total_amount#6666]

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_19.png)

In [0]:
# Example 20:
# ORDER BY / Global Sorting - Complete Example

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity > 1")

orders_df = orders_df.select("order_id","customer_id","order_date","quantity","total_amount")

result_df = orders_df.orderBy("total_amount")

result_df.write.mode("overwrite").format("noop").save()

# result_df.show()

result_df.explain()

== Physical Plan ==
Sort [total_amount#5551 ASC NULLS FIRST], true, 0
+- Exchange rangepartitioning(total_amount#5551 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=2756]
   +- Project [order_id#5546, customer_id#5547, order_date#5549, quantity#5550, total_amount#5551]
      +- Filter ((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#5565)) (_databricks_internal_edge_computed_column_skip_row#5565 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(quantity#5550)) AND (quantity#5550 > 1))
         +- FileScan parquet azure_retail_project_catalog.silver.silvers_orders[order_id#5546,customer_id#5547,order_date#5549,quantity#5550,total_amount#5551,_databricks_internal_edge_computed_column_skip_row#5565] Batched: false, DataFilters: [isnotnull(quantity#5550), (quantity#5550 > 1)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://azure-retail-project@sapracticeav.dfs.core.windows.ne

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_20.png)

In [0]:
# Example 21:
# DISTINCT / DROP DUPLICATES - Complete Example

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity > 1")

orders_df = orders_df.select("customer_id","product_id","quantity","total_amount")

distinct_df = orders_df.distinct()

result_df = distinct_df.dropDuplicates(["customer_id"])

result_df.write.mode("overwrite").format("noop").save()

result_df.explain()

== Physical Plan ==
SortAggregate(key=[customer_id#5709], functions=[finalmerge_first(merge first#5723, valueSet#5724) AS first(product_id)#5715, finalmerge_first(merge first#5727, valueSet#5728) AS first(quantity)#5717, finalmerge_first(merge first#5731, valueSet#5732) AS first(total_amount)#5719])
+- Sort [customer_id#5709 ASC NULLS FIRST], false, 0
   +- Exchange hashpartitioning(customer_id#5709, 200), ENSURE_REQUIREMENTS, [plan_id=2827]
      +- SortAggregate(key=[customer_id#5709], functions=[partial_first(product_id#5710, false) AS (first#5723, valueSet#5724), partial_first(quantity#5712, false) AS (first#5727, valueSet#5728), partial_first(total_amount#5713, false) AS (first#5731, valueSet#5732)])
         +- Sort [customer_id#5709 ASC NULLS FIRST], false, 0
            +- Project [customer_id#5709, product_id#5710, quantity#5712, total_amount#5713]
               +- Filter ((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#5745)) (_databricks_internal_edge_comp

![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_21.png)

In [0]:
# Example 22
# CACHE / PERSISTENCE - Complete Example

orders_df = spark.read.table("azure_retail_project_catalog.silver.silvers_orders")
orders_df = orders_df.filter("quantity > 1")
orders_df = orders_df.select("customer_id","product_id","quantity","total_amount")

orders_df.cache()

orders_df.write.mode("overwrite").format("noop").save()

result1_df = orders_df.groupBy("customer_id").sum("total_amount")
result1_df.write.mode("overwrite").format("noop").save()

result2_df = orders_df.filter("total_amount > 500")
result2_df.write.mode("overwrite").format("noop").save()


![](https://raw.githubusercontent.com/AyushVerma2772/dummy_databricks_github_folder/refs/heads/main/Spark%20UI/example_22.png)